# Objective 3 — Step 3: Leakage-Free Baseline Models

Run this only after Step 2 completes successfully.

This notebook evaluates six all-feature baselines on all three datasets using **5-fold StratifiedGroupKFold repeated 5 times (25 outer runs)**.

Models:
- Logistic Regression
- RBF-SVM
- KNN
- Decision Tree
- Random Forest
- XGBoost

At this stage there is **no feature selection, SMOTE, class weighting, calibration, threshold optimisation, or hyperparameter tuning**. The purpose is to establish a clean reference baseline.


In [1]:
%pip install pandas numpy scikit-learn xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: C:\Users\hp\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [2]:

from pathlib import Path
import json, math, time, warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, matthews_corrcoef, confusion_matrix,
    roc_auc_score, average_precision_score, roc_curve
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

BASE_DIR = Path(r"D:\PHD\Research Paper writing\3rd Obj. paper")
STEP2_DIR = BASE_DIR / "results" / "preprocessing_protocol"
DATA_DIR = BASE_DIR / "data" / "processed"
OUT_DIR = BASE_DIR / "results" / "baseline_models"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPEAT_SEEDS = [42, 142, 242, 342, 442]
N_FOLDS = 5

required = [
    STEP2_DIR / "preprocessing_data_dictionary.csv",
    DATA_DIR / "australian_credit_approval_cleaned.csv",
    DATA_DIR / "german_credit_cleaned.csv",
    DATA_DIR / "taiwan_credit_card_default_cleaned.csv",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Run Step 2 first. Missing files:\\n" + "\\n".join(missing)
    )

dictionary = pd.read_csv(STEP2_DIR / "preprocessing_data_dictionary.csv")

dataset_files = {
    "Australian Credit Approval": DATA_DIR / "australian_credit_approval_cleaned.csv",
    "German Credit": DATA_DIR / "german_credit_cleaned.csv",
    "Taiwan Credit Card Default": DATA_DIR / "taiwan_credit_card_default_cleaned.csv",
}

datasets = {k: pd.read_csv(v) for k, v in dataset_files.items()}

for name, df in datasets.items():
    print(name, df.shape, "adverse rate =", round(df["adverse_target"].mean(), 4))


Australian Credit Approval (690, 17) adverse rate = 0.5551
German Credit (1000, 23) adverse rate = 0.3
Taiwan Credit Card Default (30000, 27) adverse rate = 0.2212


## Recover feature roles and build fresh preprocessing inside every fold

In [3]:

def roles_for(dataset_name):
    d = dictionary[dictionary["dataset"] == dataset_name]
    return {
        role: d.loc[d["role"] == role, "variable"].astype(str).tolist()
        for role in ["categorical", "ordinal", "numerical", "identifier"]
    }

roles = {name: roles_for(name) for name in datasets}

def onehot():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False, dtype=np.float32)

def preprocessor(dataset_name, scaled):
    r = roles[dataset_name]
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", onehot()),
    ])

    if scaled:
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])
        ord_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("scaler", StandardScaler()),
        ])
    else:
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ])
        ord_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
        ])

    return ColumnTransformer([
        ("num", num_pipe, r["numerical"]),
        ("ord", ord_pipe, r["ordinal"]),
        ("cat", cat_pipe, r["categorical"]),
    ], remainder="drop")

def fresh_model(model_name):
    if model_name == "LR":
        return LogisticRegression(max_iter=2000, solver="lbfgs", random_state=42), True
    if model_name == "SVM":
        return SVC(kernel="rbf", C=1.0, gamma="scale", probability=False, random_state=42), True
    if model_name == "KNN":
        return KNeighborsClassifier(n_neighbors=5), True
    if model_name == "DT":
        return DecisionTreeClassifier(random_state=42), False
    if model_name == "RF":
        return RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1), False
    if model_name == "XGB":
        return XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9,
            objective="binary:logistic", eval_metric="logloss",
            random_state=42, n_jobs=-1, verbosity=0
        ), False
    raise ValueError(model_name)

MODEL_NAMES = ["LR", "SVM", "KNN", "DT", "RF", "XGB"]

for name in datasets:
    print(name, "predictors =", sum(len(roles[name][r]) for r in ["categorical","ordinal","numerical"]))


Australian Credit Approval predictors = 14
German Credit predictors = 20
Taiwan Credit Card Default predictors = 23


## Metric functions

In [4]:

def continuous_score(pipe, X):
    model = pipe.named_steps["model"]
    if hasattr(model, "predict_proba"):
        return pipe.predict_proba(X)[:, 1]
    return np.asarray(pipe.decision_function(X)).ravel()

def metrics(y_true, y_pred, y_score):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    gmean = math.sqrt(specificity * sensitivity) if not np.isnan(specificity+sensitivity) else np.nan

    fpr, tpr, _ = roc_curve(y_true, y_score)
    ks = float(np.max(tpr - fpr))

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_adverse": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall_adverse": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "specificity": specificity,
        "f1_adverse": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_score),
        "pr_auc": average_precision_score(y_true, y_score),
        "gmean": gmean,
        "ks_statistic": ks,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


## Run the full experiment

This is the longest cell. On the Taiwan dataset, RBF-SVM may take substantial time. Let it finish; do not interrupt unless an error occurs.


In [5]:

fold_rows = []
prediction_rows = []
split_rows = []

for dataset_name, df in datasets.items():
    print("\\n" + "="*80)
    print(dataset_name)
    print("="*80)

    r = roles[dataset_name]
    predictors = r["categorical"] + r["ordinal"] + r["numerical"]

    X = df[predictors].copy()
    y = df["adverse_target"].astype(int).copy()
    groups = df["profile_group_id"].astype(str).copy()

    for repeat_no, seed in enumerate(REPEAT_SEEDS, start=1):
        splitter = StratifiedGroupKFold(
            n_splits=N_FOLDS, shuffle=True, random_state=seed
        )

        for fold_no, (train_idx, test_idx) in enumerate(
            splitter.split(X, y, groups), start=1
        ):
            run_id = f"R{repeat_no}_F{fold_no}"

            train_groups = set(groups.iloc[train_idx])
            test_groups = set(groups.iloc[test_idx])
            shared = train_groups.intersection(test_groups)
            assert len(shared) == 0, "Profile-group leakage detected."

            split_rows.append({
                "dataset": dataset_name,
                "run_id": run_id,
                "repeat": repeat_no,
                "fold": fold_no,
                "train_records": len(train_idx),
                "test_records": len(test_idx),
                "train_adverse_rate": y.iloc[train_idx].mean(),
                "test_adverse_rate": y.iloc[test_idx].mean(),
                "shared_profile_groups": len(shared),
            })

            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            print(f"\\n{run_id}")

            for model_name in MODEL_NAMES:
                estimator, needs_scaling = fresh_model(model_name)

                pipe = Pipeline([
                    ("preprocessor", preprocessor(dataset_name, needs_scaling)),
                    ("model", estimator),
                ])

                start = time.perf_counter()
                pipe.fit(X_train, y_train)
                y_pred = pipe.predict(X_test)
                y_score = continuous_score(pipe, X_test)
                runtime = time.perf_counter() - start

                result = metrics(y_test, y_pred, y_score)
                n_transformed = len(
                    pipe.named_steps["preprocessor"].get_feature_names_out()
                )

                row = {
                    "dataset": dataset_name,
                    "run_id": run_id,
                    "repeat": repeat_no,
                    "fold": fold_no,
                    "model": model_name,
                    "train_records": len(train_idx),
                    "test_records": len(test_idx),
                    "original_predictors": len(predictors),
                    "transformed_features": n_transformed,
                    "runtime_seconds": runtime,
                    **result,
                }
                fold_rows.append(row)

                for pos, source_idx in enumerate(test_idx):
                    prediction_rows.append({
                        "dataset": dataset_name,
                        "run_id": run_id,
                        "repeat": repeat_no,
                        "fold": fold_no,
                        "model": model_name,
                        "source_row_index": int(source_idx),
                        "y_true": int(y_test.iloc[pos]),
                        "y_pred": int(y_pred[pos]),
                        "y_score": float(y_score[pos]),
                    })

                print(
                    f"{model_name}: ROC-AUC={result['roc_auc']:.4f}, "
                    f"PR-AUC={result['pr_auc']:.4f}, "
                    f"MCC={result['mcc']:.4f}, "
                    f"time={runtime:.2f}s"
                )

fold_results = pd.DataFrame(fold_rows)
predictions = pd.DataFrame(prediction_rows)
split_summary = pd.DataFrame(split_rows)

fold_results.to_csv(OUT_DIR / "baseline_fold_results_all.csv", index=False)
predictions.to_csv(OUT_DIR / "baseline_predictions_all.csv", index=False)
split_summary.to_csv(OUT_DIR / "outer_split_summary.csv", index=False)

print("\\nFull baseline experiment finished.")


\n================================================================================
Australian Credit Approval
\nR1_F1
LR: ROC-AUC=0.9194, PR-AUC=0.9451, MCC=0.7039, time=0.09s
SVM: ROC-AUC=0.9287, PR-AUC=0.9566, MCC=0.6971, time=0.09s
KNN: ROC-AUC=0.8862, PR-AUC=0.9045, MCC=0.5705, time=0.78s
DT: ROC-AUC=0.7950, PR-AUC=0.7968, MCC=0.5816, time=0.08s
RF: ROC-AUC=0.9230, PR-AUC=0.9508, MCC=0.7121, time=1.10s
XGB: ROC-AUC=0.9287, PR-AUC=0.9469, MCC=0.6826, time=0.58s
\nR1_F2
LR: ROC-AUC=0.9350, PR-AUC=0.9411, MCC=0.7380, time=0.11s
SVM: ROC-AUC=0.9216, PR-AUC=0.9287, MCC=0.7163, time=0.08s
KNN: ROC-AUC=0.9053, PR-AUC=0.9015, MCC=0.7169, time=0.07s
DT: ROC-AUC=0.8454, PR-AUC=0.8320, MCC=0.6992, time=0.06s
RF: ROC-AUC=0.9353, PR-AUC=0.9408, MCC=0.7647, time=1.16s
XGB: ROC-AUC=0.9370, PR-AUC=0.9471, MCC=0.7085, time=0.52s
\nR1_F3
LR: ROC-AUC=0.9254, PR-AUC=0.9199, MCC=0.7066, time=0.12s
SVM: ROC-AUC=0.9170, PR-AUC=0.9461, MCC=0.6965, time=0.09s
KNN: ROC-AUC=0.8811, PR-AUC=0.8703, MCC=0.6862,

## Create summary tables for the paper

In [6]:

summary = (
    fold_results
    .groupby(["dataset", "model"], as_index=False)
    .agg(
        ROC_AUC_mean=("roc_auc", "mean"),
        ROC_AUC_std=("roc_auc", "std"),
        PR_AUC_mean=("pr_auc", "mean"),
        PR_AUC_std=("pr_auc", "std"),
        Adverse_Recall_mean=("recall_adverse", "mean"),
        Adverse_Precision_mean=("precision_adverse", "mean"),
        F1_mean=("f1_adverse", "mean"),
        Balanced_Accuracy_mean=("balanced_accuracy", "mean"),
        MCC_mean=("mcc", "mean"),
        MCC_std=("mcc", "std"),
        GMean_mean=("gmean", "mean"),
        KS_mean=("ks_statistic", "mean"),
        Runtime_mean_seconds=("runtime_seconds", "mean"),
        Transformed_Features=("transformed_features", "mean"),
    )
)

numeric_cols = summary.select_dtypes(include=np.number).columns
summary[numeric_cols] = summary[numeric_cols].round(4)

summary.to_csv(OUT_DIR / "baseline_journal_table.csv", index=False)
display(summary)


,dataset,model,ROC_AUC_mean,ROC_AUC_std,PR_AUC_mean,PR_AUC_std,Adverse_Recall_mean,Adverse_Precision_mean,F1_mean,Balanced_Accuracy_mean,MCC_mean,MCC_std,GMean_mean,KS_mean,Runtime_mean_seconds,Transformed_Features
0,Australian Credit Approval,DT,0.8161,0.0320,0.7897,0.0443,0.8336,0.8363,0.8335,0.8161,0.6316,0.0637,0.8149,0.6321,0.0667,42.0
1,Australian Credit Approval,KNN,0.8969,0.0219,0.8826,0.0333,0.8742,0.8390,0.8553,0.8329,0.6698,0.0600,0.8313,0.6794,0.0994,42.0
2,Australian Credit Approval,LR,0.9274,0.0201,0.9289,0.0283,0.8552,0.8948,0.8738,0.8656,0.7275,0.0543,0.8652,0.7597,0.1066,42.0
3,Australian Credit Approval,RF,0.9308,0.0169,0.9269,0.0297,0.8753,0.8946,0.8840,0.8735,0.7449,0.0425,0.8729,0.7789,1.1262,42.0
4,Australian Credit Approval,SVM,0.9242,0.0174,0.9353,0.0186,0.8091,0.9173,0.8590,0.8599,0.7151,0.0510,0.8579,0.7610,0.0909,42.0
5,Australian Credit Approval,XGB,0.9316,0.0153,0.9328,0.0235,0.8790,0.8894,0.8833,0.8721,0.7423,0.0594,0.8717,0.7772,0.5124,42.0
6,German Credit,DT,0.6164,0.0365,0.3768,0.0573,0.4746,0.4576,0.4635,0.6164,0.2308,0.0748,0.5982,0.2327,0.0770,61.0
7,German Credit,KNN,0.7123,0.0342,0.4921,0.0614,0.3410,0.5989,0.4316,0.6215,0.2953,0.0694,0.5527,0.3430,0.0694,61.0
8,German Credit,LR,0.7838,0.0310,0.6080,0.0625,0.4869,0.6109,0.5389,0.6768,0.3797,0.0616,0.6476,0.4755,0.1455,61.0
9,German Credit,RF,0.7880,0.0203,0.6370,0.0490,0.3741,0.6976,0.4823,0.6516,0.3777,0.0598,0.5873,0.4652,1.1454,61.0


## Select the strongest clean baseline per dataset

In [7]:

best_rows = []

for dataset_name, group in summary.groupby("dataset"):
    ranked = group.sort_values(
        ["MCC_mean", "ROC_AUC_mean", "PR_AUC_mean"],
        ascending=False
    )
    best_rows.append(ranked.iloc[0])

best_baselines = pd.DataFrame(best_rows).reset_index(drop=True)
best_baselines.to_csv(
    OUT_DIR / "best_baseline_by_dataset.csv",
    index=False
)

display(best_baselines[
    [
        "dataset", "model", "ROC_AUC_mean",
        "PR_AUC_mean", "Adverse_Recall_mean",
        "F1_mean", "Balanced_Accuracy_mean", "MCC_mean"
    ]
])


,dataset,model,ROC_AUC_mean,PR_AUC_mean,Adverse_Recall_mean,F1_mean,Balanced_Accuracy_mean,MCC_mean
0,Australian Credit Approval,RF,0.9308,0.9269,0.8753,0.8840,0.8735,0.7449
1,German Credit,XGB,0.7904,0.6324,0.4783,0.5443,0.6817,0.3988
2,Taiwan Credit Card Default,XGB,0.7832,0.5605,0.3680,0.4753,0.6584,0.4029


## Final consistency checks and experiment record

In [8]:

assert len(fold_results) == 3 * 6 * 25, (
    f"Expected 450 fold-result rows, found {len(fold_results)}."
)

assert (split_summary["shared_profile_groups"] == 0).all()
assert fold_results["roc_auc"].between(0,1).all()
assert fold_results["pr_auc"].between(0,1).all()
assert fold_results["mcc"].between(-1,1).all()

config = {
    "experiment": "Objective 3 Step 3 - all-feature clean baselines",
    "positive_class": "1 = adverse/default/rejected/bad credit",
    "outer_validation": "StratifiedGroupKFold 5 folds x 5 repetitions",
    "repeat_seeds": REPEAT_SEEDS,
    "models": MODEL_NAMES,
    "feature_selection": "none",
    "imbalance_treatment": "none",
    "hyperparameter_tuning": "none",
    "calibration": "none",
    "threshold_optimization": "none",
    "primary_future_comparison_metrics": [
        "MCC", "ROC-AUC", "PR-AUC", "adverse-class F1",
        "balanced accuracy"
    ],
}

with open(
    OUT_DIR / "baseline_experiment_configuration.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(config, f, indent=4)

print("STEP 3 COMPLETED SUCCESSFULLY")
print("Output folder:", OUT_DIR)

for p in sorted(OUT_DIR.iterdir()):
    if p.is_file():
        print(" -", p.name)


STEP 3 COMPLETED SUCCESSFULLY
Output folder: D:\PHD\Research Paper writing\3rd Obj. paper\results\baseline_models
 - baseline_experiment_configuration.json
 - baseline_fold_results_all.csv
 - baseline_journal_table.csv
 - baseline_predictions_all.csv
 - best_baseline_by_dataset.csv
 - outer_split_summary.csv
